Surprisingly easy to get and run a diffusion model

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, LMSDiscreteScheduler, EulerDiscreteScheduler, EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler


# load model
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float32)
pipe = pipe.to("cuda")


In [ ]:
prompt = "A dog on a skateboard"
image = pipe(prompt).images[0]
image


Radical

Now try to do it step by step to see denosing process

In [ ]:
import matplotlib.pyplot as plt

prompt = "A dog on a skateboard"
generator = torch.Generator(device="cuda").manual_seed(0)

numSteps = 100

#use scheduler to track latent denoising
pipe.scheduler.set_timesteps(numSteps)
latents = torch.randn((1, pipe.unet.in_channels, 64, 64), generator=generator, device="cuda") #noisy square

images = []
#denoising loop
for i, t in enumerate(pipe.scheduler.timesteps):
    #predict noise and excise it (•ˋ _ ˊ•) |\
    with torch.no_grad():
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=pipe.text_encoder(pipe.tokenizer(prompt, return_tensors="pt").input_ids.to("cuda"))[0]).sample
        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

        #decode latent for preview
        if i % (numSteps / 5) == 0:
            decoded = pipe.decode_latents(latents)
            img = decoded[0]
            images.append(img)

plt.figure(figsize=(15, 5))
for i, img in enumerate(images):
    plt.subplot(1, len(images), i + 1)
    plt.imshow(img)
    plt.axis("off")
plt.show()


Now try to change the prompt partway through to get a weird amalgamation of one thing in the shape of another.

In [ ]:
promptA = "Dracula smiling"
promptB = "A dog on a skateboard"

#When to switch prompts
switchRatio = 0.5
numSteps = 100
switchStep = int(numSteps * switchRatio)

#store 'em
promptAEmb = pipe.text_encoder(pipe.tokenizer(promptA, return_tensors="pt").input_ids.to("cuda"))[0]
promptBEmb = pipe.text_encoder(pipe.tokenizer(promptB, return_tensors="pt").input_ids.to("cuda"))[0]

generator = torch.Generator(device="cuda").manual_seed(0)

#use scheduler to track latent denoising
pipe.scheduler.set_timesteps(numSteps)
latents = torch.randn((1, pipe.unet.in_channels, 64, 64), generator=generator, device="cuda") #noisy square

images = []
#denoising loop
for i, t in enumerate(pipe.scheduler.timesteps):

    if i < switchStep:
        promptEmb = promptAEmb
    else:
        promptEmb = promptBEmb

    #predict noise and excise it (•ˋ _ ˊ•) |\
    with torch.no_grad():
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=promptEmb).sample
        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

        #decode latent for preview
        if i % (numSteps / 5) == 0:
            decoded = pipe.decode_latents(latents)
            img = decoded[0]
            images.append(img)

plt.figure(figsize=(15, 5))
for i, img in enumerate(images):
    plt.subplot(1, len(images), i + 1)
    plt.imshow(img)
    plt.axis("off")
plt.show()

Turns out the image quality from the manual denoiser was far less than that of the simple one line pipe version. Because of this, most prompt switch images look like abstract art. See Dracula below to confirm he is understood by the model. Attempted increasing number of steps for denoising, but only really makes the abstractness somewhat sharper. 

In [ ]:
prompt = "Dracula smiling"
images = [pipe(prompt).images[0] for _ in range(5)]
plt.figure(figsize=(20,4))
for i, img in enumerate(images):
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")
plt.show()



In [ ]:
pipe.scheduler = LMSDiscreteScheduler.from_config(pipe.scheduler.config)
prompt = "Dracula smiling"
images = [pipe(prompt).images[0] for _ in range(5)]
plt.figure(figsize=(20,4))
for i, img in enumerate(images):
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")
plt.show()

In [ ]:
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
prompt = "Dracula smiling"
images = [pipe(prompt).images[0] for _ in range(5)]
plt.figure(figsize=(20,4))
for i, img in enumerate(images):
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")
plt.show()

In [ ]:
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
prompt = "Dracula smiling"
images = [pipe(prompt).images[0] for _ in range(5)]
plt.figure(figsize=(20,4))
for i, img in enumerate(images):
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")
plt.show()

In [ ]:
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
prompt = "Dracula smiling"
images = [pipe(prompt).images[0] for _ in range(5)]
plt.figure(figsize=(20,4))
for i, img in enumerate(images):
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")
plt.show()

Some more images using different schedulers. Time permitting we can revisit the prompt switch using one of these. 